# Урок 10 - Агенти ШІ у виробництві

У цьому уроці ви дізнаєтеся **шаблони використання в продакшені** для агентів ШІ з використанням Microsoft Agent Framework (Python). Ми розглядаємо:

- **Спостережуваність** — додавання вимірювання часу та логування взаємодій агента
- **Оцінювання** — використання агента-оцінювача для оцінки якості відповіді
- **Управління витратами** — стратегії оптимізації токенів та вибору моделі

Сценарій — **туристичний агент**, який допомагає користувачам планувати поїздки, з накладеними шарами моніторингу та оцінювання.


## Налаштування


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Виробничі міркування

Перехід агентів ШІ від прототипів до виробництва вимагає ретельної уваги до трьох ключових аспектів:

1. **Спостережуваність** — Потрібна видимість того, що робить агент, скільки часу це займає, і які інструменти він викликає. Без трасування та логування налагодження проблем у виробництві майже неможливе.

2. **Оцінювання** — Автоматизовані перевірки якості забезпечують, що відповіді агента залишаються точними, повними та корисними з часом. Агент-оцінювач може оцінювати відповіді за визначеними критеріями.

3. **Управління витратами** — Використання токенів безпосередньо впливає на вартість. Стратегії, такі як оптимізація підказок, вибір моделі та кешування допомагають тримати витрати під контролем без шкоди для якості.


## Побудова спостережуваного агента

Ми визначаємо інструменти для подорожей і обгортаємо виклик агента вимірюванням часу, щоб ми могли відстежувати затримки. У виробничому середовищі ви інтегрували б це з OpenTelemetry або подібним бекендом трасування.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Шаблони оцінювання

Поширеним підходом у виробництві є використання другого агента як **оцінювача**. Оцінювач оцінює відповідь основного агента за заздалегідь визначеними критеріями, такими як повнота, точність і корисність.

Це дає змогу:
- Автоматизовані контрольні точки якості перед тим, як відповіді потрапляють до користувачів
- Виявлення регресій при зміні підказок або моделей
- Постійний моніторинг роботи агента з часом


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегії управління витратами

Контроль витрат критично важливий для production AI агентів. Ось ключові стратегії:

| Стратегія | Опис |
|---|---|
| **Оптимізація підказок** | Тримайте системні інструкції лаконічними. Видаляйте зайвий контекст, щоб зменшити кількість вхідних токенів. |
| **Вибір моделі** | Використовуйте менші, дешевші моделі (наприклад GPT-4o-mini) для простих завдань, таких як класифікація або витягування, а більш потужні моделі залишайте для складного міркування. |
| **Кешування** | Кешуйте результати інструментів і часті запити, щоб уникнути зайвих викликів API. |
| **Бюджети токенів** | Встановлюйте ліміти `max_tokens`, щоб запобігти несподівано довгим відповідям. |
| **Пакетна обробка** | Групуйте кілька запитів користувача в один виклик API, де це можливо. |

На практиці добре працює багаторівневий підхід: направляйте прості запити до швидкої, недорогої моделі і перенаправляйте лише складніші запити на більш потужну.


## Підсумок

У цьому уроці ви дізналися, як:

1. **Додати спостережуваність** до взаємодій агента за допомогою вимірювання часу та логування, закладаючи основу для трасування та моніторингу.
2. **Оцінювати відповіді агента** автоматично за допомогою агента-оцінювача, який оцінює повноту, точність і корисність.
3. **Керувати витратами** за допомогою оптимізації підказок, вибору моделі, кешування та бюджетів токенів.

Ці методи впровадження допомагають забезпечити, що ваші агенти ШІ є надійними, вимірюваними та економічно ефективними в масштабі.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу перекладу зі штучним інтелектом [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматизовані переклади можуть містити помилки або неточності. Оригінальний документ у його вихідній мові слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний переклад, виконаний людиною. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
